# Explanation Notebook for ScenarioModel11
*Model Type:* **SVM**
*User:* UserLIB (Expertise: Intermediate, Formats: [plainText], Details: low, Purpose: general)
*Explanation Method:* LIME


## Training the SVM Model
We train a **SVM** model on the provided dataset.


In [1]:
import warnings; warnings.filterwarnings('ignore')
!pip install -q scikit-learn lime matplotlib pandas
from sklearn.datasets import load_svmlight_file
from sklearn.model_selection import train_test_split
X_sparse, y = load_svmlight_file(r"data/heart_scale")
X = X_sparse.toarray()
try:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
    print('[Info] Used stratified split to preserve class proportions in train/test.')
except Exception:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
    print('[Info] Used non-stratified split (stratification not applicable).')
class Dummy: pass
data = Dummy(); data.feature_names = ['f{}'.format(j) for j in range(X.shape[1])]
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
base_model = SVC(probability=True, random_state=0)
model = make_pipeline(StandardScaler(), base_model)
print('[Info] Normalization enabled by training policy: using StandardScaler in a pipeline.')
model.fit(X_train, y_train)
print('Accuracy: SVM = ' + format(model.score(X_test, y_test), '.2f'))


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
[Info] Used stratified split to preserve class proportions in train/test.
[Info] Normalization enabled by training policy: using StandardScaler in a pipeline.
Accuracy: SVM = 0.78


## Explaining the model using LIME
LIME explains individual predictions by fitting a simple local surrogate model around the instance.


In [2]:
import re, numpy as np, pandas as pd, matplotlib.pyplot as plt

def _get_feature_names():
    try:
        names = list(getattr(data, 'feature_names', []))
        if names and len(names) == X_train.shape[1]:
            return names
    except Exception:
        pass
    if 'df' in globals():
        cols = [c for c in df.columns if c.lower() not in ('label','target','y')]
        if len(cols) == X_train.shape[1]:
            return cols
    return [f'f{j}' for j in range(X_train.shape[1])]

RAW_NAMES = _get_feature_names()
def _humanize(name):
    s = name.replace('_',' ').strip()
    s = re.sub(r'(?<!^)(?=[A-Z])',' ', s)
    s = re.sub(r'\s+',' ', s).strip()
    return ' '.join([w.upper() if w.lower() in {'svm','pdp','ice','api','url'} else w.capitalize() for w in s.split(' ')])
HUMAN = [_humanize(n) for n in RAW_NAMES]

max_display = 3
table_rows = 6
curve_resolution = 12
ice_instances = 5
metric_digits = 2
figure_height = 3.8
line_width = 1.5
pdp_sample_rows = 6
ice_detailed_instances = 2
sampled_curve_points = 5
i = 0
print('\n[LIME overview]')
print('LIME explains one prediction by fitting a simple surrogate model around the selected instance.')
import lime.lime_tabular
explainer = lime.lime_tabular.LimeTabularExplainer(X_train, feature_names=HUMAN, class_names=['negative','positive'], mode='classification')
exp = explainer.explain_instance(X_test[i], model.predict_proba, num_features=max_display)
pairs = exp.as_list()
def _parse(rule):
    m = re.match(r'(.+?)\s*(<=|>=|<|>)\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)', rule)
    if not m:
        return {'raw_rule': rule, 'feature_label': rule, 'cmp': '', 'thr': None}
    left, cmp_op, thr = m.group(1).strip(), m.group(2), float(m.group(3))
    try:
        idx = HUMAN.index(left)
        return {'raw_rule': rule, 'feature_label': HUMAN[idx], 'cmp': cmp_op, 'thr': thr}
    except ValueError:
        return {'raw_rule': rule, 'feature_label': left, 'cmp': cmp_op, 'thr': thr}
parsed = []
for text, w in pairs:
    info = _parse(text)
    info['weight'] = float(w)
    info['direction'] = 'supports class 1' if float(w) > 0 else 'supports class 0'
    info['abs_weight'] = abs(float(w))
    parsed.append(info)
parsed = sorted(parsed, key=lambda d: abs(d['weight']), reverse=True)[:max_display]
df_lime = pd.DataFrame(parsed)
print('\n[Plain-text explanation]')
print('This explanation summarizes the most influential factors for the selected prediction.')
for _, r in df_lime.iterrows():
    print(' - ' + str(r['feature_label']) + ': ' + str(r['direction']) + ' | weight=' + format(r['weight'], '.4f'))



[LIME overview]
LIME explains one prediction by fitting a simple surrogate model around the selected instance.

[Plain-text explanation]
This explanation summarizes the most influential factors for the selected prediction.
 - F2: supports class 0 | weight=-0.1794
 - F7: supports class 0 | weight=-0.0870
 - F0: supports class 0 | weight=-0.0662
